In [ ]:
# =====================================================
# DOCUMENT EXTRACTION + SEMANTIC SEARCH
# =====================================================
!pip install -q pymupdf sentence-transformers faiss-cpu

# =====================================================
# IMPORTS
# =====================================================

import fitz
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# =====================================================
# SPECIFY YOUR DOCUMENT HERE
# =====================================================

PDF_FILE = "RAG_Fundamentals.pdf"

# =====================================================
# EXTRACT TEXT FROM PDF
# =====================================================

def extract_text_from_pdf(pdf_path):

    document = fitz.open(pdf_path)

    text = ""

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        text += page.get_text()

    document.close()

    return text

print("Extracting text from PDF...")

document_text = extract_text_from_pdf(PDF_FILE)

print("Text Extraction Complete")
print("Total Characters:", len(document_text))

# =====================================================
# CHUNKING
# =====================================================

def create_chunks(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i + chunk_size])

    return chunks

chunks = create_chunks(document_text)

print("Total Chunks:", len(chunks))

# =====================================================
# LOAD EMBEDDING MODEL
# =====================================================

print("Loading Embedding Model...")

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# =====================================================
# CREATE EMBEDDINGS
# =====================================================

print("Generating Embeddings...")

embeddings = model.encode(
    chunks,
    show_progress_bar=True
)

embeddings = np.array(
    embeddings
).astype("float32")

print("Embeddings Created Successfully")

# =====================================================
# CREATE FAISS VECTOR DATABASE
# =====================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS Index Created")
print("Vectors Stored:", index.ntotal)

# =====================================================
# SEMANTIC SEARCH FUNCTION
# =====================================================

def search_document(query, top_k=1):

    query_embedding = model.encode([query])

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx])

    return retrieved_chunks

# =====================================================
# ASK QUESTIONS
# =====================================================

while True:

    question = input("\nAsk a Question (type 'exit' to stop): ")

    if question.lower() == "exit":
        print("Session Ended")
        break

    results = search_document(
        question,
        top_k=3
    )

    print("\nTop Matching Results:\n")

    for i, result in enumerate(results, start=1):

        print(f"\nResult {i}")
        print("-" * 60)
        print(result[:1000])

    print("\n" + "=" * 80)

Extracting text from PDF...
Text Extraction Complete
Total Characters: 14854
Total Chunks: 30
Loading Embedding Model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating Embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings Created Successfully
FAISS Index Created
Vectors Stored: 30

Top Matching Results:


Result 1
------------------------------------------------------------
 Weaviate, Chroma).
7. RAG Support
●
Built-in support for Retrieval-Augmented Generation (connects LLMs to external data).
Use Cases
●Chatbots & Virtual Assistants
●Q&A systems over internal documents
●Code generation assistants
●Summarization & report generation
●AI Agents for workflow automation
What is LlamaIndex?
LlamaIndex is a data framework that helps connect Large Language Models (LLMs) (like 
GPT-4, Claude, LLaMA) with your private or custom data.
It is especially useful for Retrieval-A


Top Matching Results:


Result 1
------------------------------------------------------------
 Weaviate, Chroma).
7. RAG Support
●
Built-in support for Retrieval-Augmented Generation (connects LLMs to external data).
Use Cases
●Chatbots & Virtual Assistants
●Q&A systems over internal documents
●Code generation assistants
●Summari